# 16 — CatBoost com Categóricas Nativas (sem one-hot)
## PRT Seguros

Até agora alimentamos o CatBoost com todas as colunas já em one-hot (`segmento_Bronze`,
`segmento_Ouro`, ... como colunas 0/1 separadas). Isso desperdiça o principal diferencial do
CatBoost: ele tem um algoritmo próprio (*ordered target statistics*) pra lidar com colunas
categóricas cruas, que costuma capturar mais sinal — e ser mais robusto a shift — do que one-hot
manual, especialmente quando há muitas categorias.

**O que fazemos aqui:** desfazemos o one-hot (reconstruímos a coluna categórica original a partir
das dummies) para `segmento`, `veiculo`, `canal`, `canal_aquisicao`, `metodo_pagamento`,
`escolaridade`, `tipo_cobertura`, `regiao`, e tratamos `cluster` também como categórico (é um ID de
grupo, não uma quantidade ordinal). Passamos essas colunas para o CatBoost via `cat_features`.

## 1. Carregar dados (versão base, com todas as colunas, antes do one-hot)

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready.csv")
val = pd.read_csv("dados_processados/val_model_ready.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready.csv")
print(f"Treino: {train.shape} | Val: {val.shape} | Kaggle: {kaggle.shape}")


Treino: (80000, 74) | Val: (20000, 74) | Kaggle: (100000, 73)


## 2. Reconstruir as colunas categóricas originais a partir das dummies

`idxmax(axis=1)` pega, entre as colunas do grupo, qual está marcada com 1 — e vira o nome da
categoria (removendo o prefixo). Como cada grupo sempre tem uma dummy `_NaN` explícita, não sobra
nenhum valor realmente faltante — o "faltante" vira só mais uma categoria, que é exatamente o que
o CatBoost sabe tratar bem.

In [2]:
GRUPOS_CATEGORICOS = {
    "escolaridade": ["escolaridade_Fundamental", "escolaridade_Medio", "escolaridade_NaN",
                      "escolaridade_Pos", "escolaridade_Superior"],
    "canal_aquisicao": ["canal_aquisicao_Agente", "canal_aquisicao_Digital", "canal_aquisicao_Indicacao",
                         "canal_aquisicao_NaN", "canal_aquisicao_Telefone"],
    "canal": ["canal_Carta", "canal_Email", "canal_Nao Informado", "canal_Telefone", "canal_WhatsApp"],
    "metodo_pagamento": ["metodo_pagamento_NaN", "metodo_pagamento_boleto", "metodo_pagamento_credito",
                          "metodo_pagamento_debito", "metodo_pagamento_pix"],
    "tipo_cobertura": ["tipo_cobertura_NaN", "tipo_cobertura_basica", "tipo_cobertura_padrao",
                        "tipo_cobertura_premium"],
    "segmento": ["segmento_Bronze", "segmento_Diamante", "segmento_NaN", "segmento_Ouro", "segmento_Prata"],
    "veiculo": ["veiculo_Carro", "veiculo_Moto", "veiculo_NaN", "veiculo_Pickup", "veiculo_SUV", "veiculo_Van"],
    "regiao": ["regiao_Centro-Oeste", "regiao_NaN", "regiao_Nordeste", "regiao_Sudeste", "regiao_Sul"],
}

def reconstruir_categoricas(df):
    df = df.copy()
    for grupo, cols in GRUPOS_CATEGORICOS.items():
        prefixo = grupo + "_"
        categoria = df[cols].idxmax(axis=1).str.replace(prefixo, "", regex=False)
        df[grupo] = categoria
        df = df.drop(columns=cols)
    df["cluster"] = df["cluster"].astype(int).astype(str)
    return df

train_cat = reconstruir_categoricas(train)
val_cat = reconstruir_categoricas(val)
kaggle_cat = reconstruir_categoricas(kaggle)

COLUNAS_CATEGORICAS = list(GRUPOS_CATEGORICOS.keys()) + ["cluster"]
print("Exemplo de categorias reconstruídas:")
print(train_cat[COLUNAS_CATEGORICAS].head())
print()
for c in COLUNAS_CATEGORICAS:
    print(f"{c:<18} valores únicos: {sorted(train_cat[c].unique())}")


Exemplo de categorias reconstruídas:
  escolaridade canal_aquisicao     canal metodo_pagamento tipo_cobertura  \
0        Medio          Agente     Email           boleto         basica   
1          NaN       Indicacao     Email           debito         basica   
2     Superior       Indicacao  Telefone              pix        premium   
3          Pos       Indicacao  Telefone          credito        premium   
4     Superior        Telefone     Carta              NaN        premium   

   segmento veiculo        regiao cluster  
0     Prata     Van  Centro-Oeste       1  
1      Ouro    Moto       Sudeste       4  
2    Bronze  Pickup       Sudeste       3  
3      Ouro    Moto  Centro-Oeste       2  
4  Diamante   Carro  Centro-Oeste       2  

escolaridade       valores únicos: ['Fundamental', 'Medio', 'NaN', 'Pos', 'Superior']
canal_aquisicao    valores únicos: ['Agente', 'Digital', 'Indicacao', 'NaN', 'Telefone']
canal              valores únicos: ['Carta', 'Email', 'Nao Informa

## 3. Treinar CatBoost passando `cat_features` (sem one-hot manual)

In [3]:
feature_cols = [c for c in train_cat.columns if c not in (ID_COL, TARGET_COL)]
cat_idx = [feature_cols.index(c) for c in COLUNAS_CATEGORICAS]

X_train, y_train = train_cat[feature_cols], train_cat[TARGET_COL]
X_val, y_val = val_cat[feature_cols], val_cat[TARGET_COL]
X_kaggle = kaggle_cat[feature_cols]

X_tr_es, X_es, y_tr_es, y_es = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)
neg_pos_ratio_es = (y_tr_es == 0).sum() / (y_tr_es == 1).sum()

MELHORES_PARAMS_CAT = {"random_strength": 0.5, "learning_rate": 0.02, "l2_leaf_reg": 9, "depth": 6}

pool_tr = Pool(X_tr_es, y_tr_es, cat_features=cat_idx)
pool_es = Pool(X_es, y_es, cat_features=cat_idx)
pool_val = Pool(X_val, cat_features=cat_idx)
pool_kaggle = Pool(X_kaggle, cat_features=cat_idx)

cat_nativo = CatBoostClassifier(
    iterations=3000, **MELHORES_PARAMS_CAT, scale_pos_weight=neg_pos_ratio_es,
    eval_metric="AUC", random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=200,
)
cat_nativo.fit(pool_tr, eval_set=pool_es)
print(f"Melhor iteração: {cat_nativo.get_best_iteration()}")

proba_val_nativo = cat_nativo.predict_proba(pool_val)[:, 1]
auc_nativo = roc_auc_score(y_val, proba_val_nativo)
print(f"\nCatBoost categóricas NATIVAS : AUC-ROC val = {auc_nativo:.4f}")
print(f"CatBoost one-hot (v2, tuned) : AUC-ROC val = 0.8263  (referência)")


0:	test: 0.8127039	best: 0.8127039 (0)	total: 309ms	remaining: 15m 25s


200:	test: 0.8307023	best: 0.8307279 (199)	total: 20.3s	remaining: 4m 43s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8311952903
bestIteration = 310

Shrink model to first 311 iterations.
Melhor iteração: 310

CatBoost categóricas NATIVAS : AUC-ROC val = 0.8264
CatBoost one-hot (v2, tuned) : AUC-ROC val = 0.8263  (referência)


## 4. Gerar submissão

In [4]:
proba_kaggle_nativo = cat_nativo.predict_proba(pool_kaggle)[:, 1]
submissao = pd.DataFrame({"Id": kaggle_cat[ID_COL], "probabilidade_churn": proba_kaggle_nativo})
submissao.to_csv("submissions/submission_catboost_categoricas_nativas.csv", index=False)
print("Salvo: submissions/submission_catboost_categoricas_nativas.csv")


Salvo: submissions/submission_catboost_categoricas_nativas.csv
